<div dir="ltr" style="text-align:left">

# Day 6 Lab — Is the Warming Real, or Just Weather?

**Client:** a friend who insists *"it was cold last week, so global warming is a myth."*
**Mission:** answer with 146 years of data, not opinion — separate the **signal** (the long-run trend) from the **noise** (year-to-year weather).
**Deliverable:** one chart showing the trend, and a one-line verdict you can defend.

---

Two working rules: write your prediction before you run any reveal cell, and put units on every axis. Bracketed codes like `[P-6]` point to the pandas/matplotlib cheat sheets in *Foundations of Evidence*.

</div>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ── Opening ritual — run this cell as-is ──
BASE = "https://raw.githubusercontent.com/Saleh314Astro/data-science-journey-data/main/"

def load(name, folder="clean"):
    """Read a dataset from GitHub [P-1]; fall back to the local copy if offline."""
    try:
        return pd.read_csv(BASE + folder + "/" + name, parse_dates=["date"])
    except Exception:
        return pd.read_csv("../data/" + folder + "/" + name, parse_dates=["date"])

temp = load("gistemp_monthly.csv")   # NASA GISTEMP monthly anomaly, deg C
temp.head()   # [P-2]

<div dir="ltr" style="text-align:left">

The data is already an **anomaly** — each value is degrees Celsius above or below the 1951–1980 average. Zero means "the old normal." Positive means "warmer than normal."

</div>

<div dir="ltr" style="text-align:left">

### Predict before you run
You are about to plot all 1,758 monthly values from 1880 to today. Will the line be a clean rising curve, or something messier? Sketch what you expect.

**My written prediction:** *(double-click and type here before running the next cell)*

…

</div>

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))            # [M-1] [M-6]
ax.plot(temp["date"], temp["value"], lw=0.5, color="gray", label="Monthly anomaly")   # [M-2]
ax.axhline(0, color="black", lw=0.8)               # [M-7]  the old normal
ax.set_xlabel("Year")
ax.set_ylabel("Temperature anomaly (deg C)")       # [M-4]
ax.set_title("NASA GISTEMP, every month 1880-2026")
ax.legend()                                        # [M-5]
plt.show()

<div dir="ltr" style="text-align:left">

### Investigate — The raw plot
1. Describe the line in one sentence. Is it noisy, smooth, or both?
2. A single cold month sits below a warm neighbour. Can one month — or one year — settle the warming question? Why not?
3. In climate science, **weather** is this week's noise and **climate** is the 30-year signal. Which one is the friend's 'it was cold last week' argument about?

*Answer in this cell, under each question.*

</div>

<div dir="ltr" style="text-align:left">

### The new skill today — the moving average (`rolling`)
To hear the signal under the noise, average a sliding window of months. A 10-year window is 120 months. `rolling(window=120)` [P-6] walks that window along the series and averages each position — smoothing the jitter and exposing the trend.

</div>

<div dir="ltr" style="text-align:left">

### Predict before you run
What will a 120-month (10-year) moving average do to the jagged line? Will it move the overall level up/down, or just smooth the wiggles?

**My written prediction:** *(double-click and type here before running the next cell)*

…

</div>

In [ ]:
# YOUR TASK — the new skill [P-6]:
# make a column temp["smooth"] = the 120-month moving average of temp["value"]
# YOUR CODE HERE


# ── Safety net: if you have not written the task above yet, this runs it
# so the rest of the notebook still works. Try it yourself first.
if "smooth_done" not in dir():
    temp["smooth"] = temp["value"].rolling(window=120, center=True).mean()
    smooth_done = True

temp[["date", "value", "smooth"]].iloc[600:605]

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(temp["date"], temp["value"], lw=0.5, color="lightgray", label="Monthly (noise)")   # [M-2]
ax.plot(temp["date"], temp["smooth"], lw=2.2, color="crimson", label="10-year average (signal)")
ax.axhline(0, color="black", lw=0.8)          # [M-7]
ax.set_xlabel("Year")
ax.set_ylabel("Temperature anomaly (deg C)")
ax.set_title("Signal vs noise: the same data, read honestly")
ax.legend()
plt.show()

<div dir="ltr" style="text-align:left">

### Investigate — Signal vs noise
1. Compare this chart to the first one. What did the moving average reveal that the raw plot hid?
2. Now answer the client: is the warming real, or just weather? Write the one-line verdict you would defend.
3. The smooth line is blank at the very start and very end. Why does a centred 10-year window have nothing to show there?

*Answer in this cell, under each question.*

</div>

<div dir="ltr" style="text-align:left">

### One more number — mean and anomaly by hand
You built the **mean** by hand in Week 1. Use it here to put a number on the change: compare the average anomaly of the 1880s decade with the 2010s decade.

</div>

In [ ]:
early = temp.loc[(temp["date"] >= "1880-01-01") & (temp["date"] < "1890-01-01"), "value"].mean()   # [P-5] [P-12]
recent = temp.loc[(temp["date"] >= "2010-01-01") & (temp["date"] < "2020-01-01"), "value"].mean()
print(f"1880s average anomaly:  {early:+.2f} deg C")
print(f"2010s average anomaly:  {recent:+.2f} deg C")
print(f"Change between the two decades: {recent - early:+.2f} deg C")

<div dir="ltr" style="text-align:left">

### Stretch — reproduce the clean file from the raw one
You loaded a tidy two-column file. But NASA does not publish it that way. Open the untouched original `data/raw/GLB.Ts+dSST.csv` and rebuild `gistemp_monthly.csv` yourself — the real work of a data scientist.

</div>

In [ ]:
# STRETCH — rebuild the clean file from the raw NASA table.
# The raw file: 1 title line to skip, months in columns Jan..Dec,
# missing values marked "***". Goal: a tidy (date, value) frame.
def load_raw(name):
    try:
        return pd.read_csv(BASE + "raw/" + name, skiprows=1, na_values="***")
    except Exception:
        return pd.read_csv("../data/raw/" + name, skiprows=1, na_values="***")

raw = load_raw("GLB.Ts+dSST.csv")
months = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
# YOUR CODE HERE: keep Year + the 12 month columns, melt to long form,
# build a "date" column, drop missing values, sort by date.


# ── Safety net: if you have not written the task above yet, this runs it
# so the rest of the notebook still works. Try it yourself first.
if "clean_check" not in dir():
    long = raw[["Year"] + months].melt(id_vars="Year", var_name="m", value_name="value")
    long["month"] = long["m"].map({m: i+1 for i, m in enumerate(months)})
    long = long.dropna(subset=["value"])
    long["date"] = pd.to_datetime(dict(year=long["Year"], month=long["month"], day=1))
    rebuilt = long.sort_values("date")[["date", "value"]].reset_index(drop=True)
    clean_check = rebuilt

print("rows rebuilt:", len(clean_check))
clean_check.head()

<div dir="ltr" style="text-align:left">

### Closing reflection — Day 6
A planet held its balance within a single degree for 150 years, and you just watched it start to drift. Which felt more convincing — the raw data or the moving average — and why?

**My reflection:** *(three lines at most)*

</div>